In [ ]:
%run ../../langchain_common.py

## Summarize messages

In [ ]:
summarization_model = create_noreason_llm(model="databricks-gemma-3-12b")
pg_checkpointer = create_pg_checkpointer()


agent = create_agent(
    model=llm_noreason, 
    checkpointer=pg_checkpointer,
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            trigger=("tokens", 100),
            keep=("messages", 1)
        )
    ],
)

In [ ]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": f"{new_conversation_id()}"}}
)

pprint(response)

In [ ]:
print(response["messages"][0].content)

## Trim/delete messages

In [ ]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    filtered_messages = [RemoveMessage(id=m.id) for m in tool_messages]
    return {"messages": filtered_messages}

In [ ]:
agent = create_agent(
    model=llm_noreason,
    checkpointer=pg_checkpointer,
    middleware=[trim_messages],
)

In [ ]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": f"{new_conversation_id()}"}}
)

for msg in response["messages"]:
    print(f"{msg.type}: {msg.content}")

`trim_messages` is a LangChain utility that shortens a chat history so it fits a token budget before sending it to a model.

In this cell, it is configured as:

- `max_tokens=65`: target limit for total message tokens.
- `strategy="last"`: keep the **most recent** messages and drop older ones first.
- `token_counter="approximate"`: use a fast approximate token count (not exact model tokenization).
- `include_system=True`: preserve `SystemMessage` content when possible.
- `allow_partial=False`: do **not** keep partial/truncated messages; each kept message must be whole.
- `start_on="human"`: trimming boundary starts from a human turn, helping keep conversation alignment by beginning with user context.

### What the `"last"` strategy means
With `"last"`, the trimmer walks backward from the end of the conversation and keeps the newest exchanges until adding another message would exceed `max_tokens`. This is useful when recent context is more important than older history.

In [ ]:
from langchain_core.messages import SystemMessage, trim_messages

trimmer = trim_messages(
    max_tokens=65,
    strategy="last", 
    token_counter="approximate",
    include_system=True,
    allow_partial=False, 
    start_on="human", 
)

messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="what's 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]

trimmer.invoke(messages)

## `filter_messages` in LangChain

`filter_messages` selects only the messages you want from a chat history and returns a new list.

- Useful for cleaning context before sending messages to a model.
- Can filter by **message type** (e.g., `"human"`, `"ai"`, `"system"`).
- Can also filter using metadata like names/IDs (when provided).

### Quick example
```python
filter_messages(messages, include_types="human")
```

This keeps only `HumanMessage` entries and removes system/AI messages.


In [ ]:
from langchain_core.messages import SystemMessage, filter_messages, HumanMessage, AIMessage

messages = [
    SystemMessage("you are a good assistant", id="1"),
    HumanMessage("example input", id="2", name="example_user"),
    AIMessage("example output", id="3", name="example_assistant"),
    ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
    HumanMessage("real input", id="4", name="bob"),
    AIMessage("real output", id="5", name="alice"),
]


In [ ]:

filter_messages(messages, include_types="human")

In [ ]:
filter_messages(messages, include_types={"human", "tool"})

### Rebuilding our custom middleware using langchain methods ...

In [ ]:
@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]
    filtered_messages = filter_messages(messages, include_types={"human", "tool"})

    return {"messages": filtered_messages}